# 04 — Analysis: from Gold to the two headline findings

This notebook computes nothing new. Everything here reads `gold.bid.*` tables
produced by `03_gold_bid_performance` and renders the two comparisons the
README leads with, plus the data-quality context that qualifies them.

Both this notebook and the README are generated from the same seeded run
(`--seed 42`), so the headline percentages should match. Where a figure here
involves a judgement call (e.g. which loss reasons count as price-related),
the classification is made explicit in the cell rather than assumed.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Aggregates only — every gold table here is small enough (a handful of
# rows per dimension) that a single-node toPandas() is the right call.
# Pulling the underlying fact table to the driver would not be.
overall            = spark.table("gold.bid.performance_overall").toPandas()
by_channel         = spark.table("gold.bid.performance_by_channel").toPandas()
by_executive       = spark.table("gold.bid.performance_by_account_executive").toPandas()
by_segment         = spark.table("gold.bid.performance_by_segment").toPandas()
by_value_band      = spark.table("gold.bid.performance_by_value_band").toPandas()
loss_coverage      = spark.table("gold.bid.loss_reason_coverage").toPandas()
loss_reasons       = spark.table("gold.bid.loss_reasons").toPandas()
open_pipeline      = spark.table("gold.bid.open_pipeline").toPandas()

## Finding 1 — the migration artefact

Bulk-loaded bids convert at roughly a third of the rate of organically
entered ones. The gap is not noise — it is 58% of the dataset behaving like
a different population, concentrated in one executive's portfolio.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel 1: bulk vs organic, company-wide -------------------------------
channel_labels = by_channel["is_bulk_load"].map({True: "Bulk-loaded", False: "Organic"})
axes[0].bar(channel_labels, by_channel["win_rate_by_count"],
            color=["#c0c0c0", "#2b6cb0"])
axes[0].set_title("Win rate: bulk vs organic")
axes[0].set_ylabel("Win rate")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_channel["win_rate_by_count"]):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center")

# --- Panel 2: Executive 4, all bids vs organic-only ------------------------
exec4 = by_executive[by_executive["account_executive"] == "Executive 4"].iloc[0]
company_avg = overall["win_rate_by_count"].iloc[0]

bars = axes[1].bar(
    ["All bids", "Organic only", "Company\naverage"],
    [exec4["wr_all"], exec4["wr_organic"], company_avg],
    color=["#c0392b", "#2b6cb0", "#888888"],
)
axes[1].set_title("Executive 4: worst performer, or artefact?")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, [exec4["wr_all"], exec4["wr_organic"], company_avg]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/finding1_migration_artefact.png", bbox_inches="tight")
plt.show()

print(f"Executive 4 — all bids: {exec4['wr_all']:.1f}%   organic only: {exec4['wr_organic']:.1f}%   "
      f"gap: {exec4['artefact_gap']:.1f}pp")

### Caveat this chart cannot show

The bulk pool was drawn concentrated in Financial Services / Public Sector —
the two lowest-baseline-win-rate segments in the dataset. Removing bulk load
moves Executive 4 from worst to above-average, but part of what's left is
still segment mix, not entirely a clean organic signal. A logistic
regression on `outcome ~ is_bulk_load + segment + value_quartile` would
isolate the two effects properly — noted as a follow-up, not done here, so
this chart is not overstated as more rigorous than it is.

## Finding 2 — win rate by count vs by value

The company wins small contracts and loses large ones. Counting bids
flatters performance; weighting by revenue does not.

In [0]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(by_value_band["value_quartile"].astype(str), by_value_band["win_rate_by_count"],
       color="#2b6cb0")
ax.set_title("Win rate by contract-value quartile")
ax.set_xlabel("Value quartile (1 = smallest, 4 = largest)")
ax.set_ylabel("Win rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_value_band["win_rate_by_count"]):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/finding2_value_effect.png", bbox_inches="tight")
plt.show()

gap = overall["win_rate_by_count"].iloc[0] - overall["win_rate_by_value"].iloc[0]
print(f"Win rate by count: {overall['win_rate_by_count'].iloc[0]:.1f}%   "
      f"by value: {overall['win_rate_by_value'].iloc[0]:.1f}%   gap: {gap:.1f}pp")

## Why it happens: loss reasons, and how little of the picture they cover

The reason field would settle whether Finding 2 is a pricing problem. It's
populated for a single-digit share of losses — reported here as a signal,
explicitly not a population estimate.

In [0]:
coverage_pct = loss_coverage["coverage_pct"].iloc[0]
print(f"Loss reason coverage: {coverage_pct:.1f}% "
      f"({loss_coverage['losses_with_reason'].iloc[0]} of {loss_coverage['losses_total'].iloc[0]} losses)")
print(f"Attributed to the placeholder competitor: {loss_coverage['placeholder_attributed'].iloc[0]}")

# Explicit exclusion list rather than a keyword regex: with only a
# handful of distinct reasons, naming the non-price ones directly is more
# reliable than a substring match (e.g. "Financial proposal not
# competitive" is price-related but contains none of cost/price/commercial).
NON_PRICE_REASONS = {
    "Bid cancelled by client",
    "Scope did not match expectations",
    "Incumbent held established relationship",
    "Bid suspended / postponed",
    "Insufficient information from client",
    "Reason not recorded",
}
price_related = loss_reasons[~loss_reasons["loss_reason"].isin(NON_PRICE_REASONS)]
price_share = price_related["share_pct"].sum()
print(f"Price/cost-related share of recorded reasons: {price_share:.1f}%")

loss_reasons.sort_values("losses", ascending=False).head(10)

## Open pipeline

Reported on its own — never folded into the loss column, which would
understate the win rate.

In [0]:
open_pipeline

## What this notebook does not claim

See the README's "What this analysis cannot tell you" section for the full
list — sales-cycle length, the non-random loss-reason sample, monthly vs
total contract value, and the absence of bid cost. Repeating it here would
just be duplication; the point is that those limitations apply to every
chart above, not only to the text they sit next to.